# Malaria model selection: the gate

The human step between a ranked selection run and the model the forecast loads
(`.claude/SELECTION_PIPELINE_PLAN.md` step 3; `docs/model_selection/RUNBOOK.md` step 4).
This notebook is **import-only**: every action is one call into
`idd_forecast_mbp.select.gate`, which is tested. Run it with the repo venv kernel from this
directory.

1. Open the run (the config's `run_dir`, or an override) and read the tentative pick.
2. Move the rank-time toggles; the pick and its candidates re-rank live.
3. **Record pick** writes `selection_result.json` (status `recorded`, with the toggle values in
   force), re-renders `report.html`, and runs the final fit into the models node's `working/`.
4. **Flag best** freezes that fit as a snapshot and promotes it (`current` = the model the
   forecast loads). Enabled only once a fit whose `run.json` points at this result exists.

No path is typed here: the R image and shell come from the environment
(`MBP_R_IMAGE`, `MBP_R_SHELL`) or the text boxes below.


In [ ]:
import os
from pathlib import Path

import ipywidgets as w
import pandas as pd
from IPython.display import display

from idd_forecast_mbp import constants as mbpc
from idd_forecast_mbp.select import gate
from idd_forecast_mbp.select.rank import format_pick

REPO = Path.cwd().resolve().parents[1]
CONFIG = REPO / "reports" / "model_selection" / "malaria_selection_config.yaml"
RUN_DIR = None  # None -> the config's run_dir; or an absolute path / a path relative to the 03-modeling_data root

session = gate.GateSession.open(CONFIG, RUN_DIR)
base = session.config.rank
print(f"run dir : {session.run_dir}")
print(f"specs   : {len(session.summary)}")
print(f"config  : {CONFIG}")
print(format_pick(session.rerank()))

## Toggles
Only these keys may change here (`gate.OVERRIDABLE`); everything else is the committed config.

In [ ]:
tolerance = w.FloatSlider(value=base.tolerance_value, min=0.0, max=1.0, step=0.01,
                          description="tolerance (SD frac)", readout_format=".2f", continuous_update=False)
focus = w.Dropdown(options=["topsis_score", "borda_score"], value=base.focus_metric, description="focus metric")
tau = w.FloatSlider(value=base.tau_prune_threshold, min=0.5, max=1.0, step=0.01,
                    description="tau prune", readout_format=".2f", continuous_update=False)
top_n = w.IntSlider(value=base.profile_top_n, min=1, max=50, description="profile top n")
cull = w.Checkbox(value=base.cull_nonconverged, description="cull non-converged")
out = w.Output()

state = {"result": None}

def overrides():
    return dict(tolerance_value=float(tolerance.value), focus_metric=focus.value,
                tau_prune_threshold=float(tau.value), profile_top_n=int(top_n.value),
                cull_nonconverged=bool(cull.value))

def rerank(_=None):
    with out:
        out.clear_output(wait=True)
        try:
            res = session.rerank(**overrides())
        except ValueError as exc:
            print(f"refused: {exc}")
            state["result"] = None
            return
        state["result"] = res
        print(format_pick(res))
        cols = [c for c in ["spec_index", "n_terms", "n_smooths", "n_scams", res.params.focus_metric, "formula_text"]
                if c in res.candidates.columns]
        display(res.candidates[cols].reset_index(drop=True))

for widget in (tolerance, focus, tau, top_n, cull):
    widget.observe(rerank, names="value")
display(w.VBox([tolerance, focus, tau, top_n, cull]), out)
rerank()

## Record pick, fit, flag best

In [ ]:
r_image = w.Text(value=os.environ.get("MBP_R_IMAGE", ""), description="R image", layout=w.Layout(width="70%"))
r_shell = w.Text(value=os.environ.get("MBP_R_SHELL", ""), description="R shell", layout=w.Layout(width="70%"))
description = w.Text(value="", description="freeze why", placeholder="one line: why this model becomes current", layout=w.Layout(width="70%"))
label = w.Text(value="", description="label", placeholder="optional, e.g. full_model_selection_2026_10", layout=w.Layout(width="70%"))
record_btn = w.Button(description="Record pick + fit", button_style="warning")
flag_btn = w.Button(description="Flag best", button_style="danger", disabled=True)
log = w.Output()

def refresh_flag_button():
    st = gate.fit_status(mbpc.MAL_MODELS_NODE, session.result_path)
    flag_btn.disabled = not st.matches
    flag_btn.tooltip = f"fit: {st.location} {st.path or ''}".strip()
    return st

def on_record(_):
    with log:
        log.clear_output(wait=True)
        res = state["result"]
        if res is None:
            print("no ranking to record (a toggle combination was refused)"); return
        if not (r_image.value and r_shell.value):
            print("R image and shell are required to fit"); return
        paths = session.record_pick(res)
        for k, p in paths.items():
            print(f"wrote {k}: {p}")
        argv = gate.fit_command(CONFIG, session.run_dir, r_image=r_image.value, r_shell=r_shell.value)
        print("fitting:", " ".join(argv))
        gate.launch_fit(argv)
        st = refresh_flag_button()
        print(f"fit: {st.location} -> {st.path}; Flag best {'enabled' if st.matches else 'disabled'}")

def on_flag(_):
    with log:
        if not description.value.strip():
            print("a freeze description is required"); return
        name = gate.flag_best(mbpc.MAL_MODELS_NODE, session.result_path,
                              description=description.value.strip(), label=label.value.strip() or None)
        print(f"promoted snapshot {name} on {mbpc.MAL_MODELS_NODE}; the forecast now loads it")
        refresh_flag_button()

record_btn.on_click(on_record)
flag_btn.on_click(on_flag)
st = refresh_flag_button()
print(f"existing fit for this result: {st.location} {st.path or ''}")
display(w.VBox([r_image, r_shell, description, label, w.HBox([record_btn, flag_btn]), log]))